> **Course: AIML ZG528 AI and ML for Robotics**

>  **Author: Raja vadhana Prabhakar**
raja.vadhana@pilani.bits-pilani.ac.in

> **Objective:**
To implement AIML technique as a steering function in path planning.

**About:**
Design:
  > State : 4 dimensional vector : [dx, dy, distanceToGoal, distanceToNearestObstacle]

  >  Action : Continous value in 2D [-1, +1]

  >  Output : Policy that maps State to Action oriented toward Goal.

### Install dependencies. Its a one time activity!

In [1]:
# Colab cell 1: install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install -q gymnasium pybullet imageio[ffmpeg] matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [95]:
# Put this near top of file (once)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


### Import necessary Libraries

In [96]:
import os
import time
import math
import random
from collections import namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pybullet as p
import pybullet_data
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display
import gymnasium as gym
from gymnasium import spaces
import numpy as np


## Function Definitions

### DRL : PPO Agent to handle continous action space

In [97]:
class PPOAgent(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim) - 0.5)
        #self.log_std = nn.Parameter(torch.full((action_dim,), -0.5, dtype=torch.float32))


        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def get_action(self, obs):
        # obs: numpy array or torch tensor (1D)
        if isinstance(obs, np.ndarray):
            obs = torch.tensor(obs, dtype=torch.float32)
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        obs = obs.to(next(self.parameters()).device)  # ensure obs on same device as model
        mean = self.actor(obs)
        std = torch.exp(self.log_std).unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        action = dist.rsample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        value = self.critic(obs).squeeze(-1)
        # return numpy action for env, but keep log_prob and value as tensors on device
        return action.detach().cpu().numpy()[0], log_prob.detach(), value.detach()

    def evaluate_actions(self, obs_batch, action_batch):
        mean = self.actor(obs_batch)

        log_std = torch.clamp(self.log_std, min=-20.0, max=2.0)
        std = torch.exp(log_std).unsqueeze(0).expand_as(mean)
        #std = torch.exp(self.log_std).unsqueeze(0).expand_as(mean)
        dist = torch.distributions.Normal(mean, std)
        log_probs = dist.log_prob(action_batch).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        values = self.critic(obs_batch).squeeze(-1)
        return log_probs, entropy, values



In [99]:
# Put this near top of file (once)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## Wrapper functions to be compatible with Gymnasium Environment

In [137]:
class PyBulletPointEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 60}

    def __init__(self, start=(1.0,1.0), goal=(10.0,10.0), obstacles=None, map_size=(15.0,15.0),
                 time_step=1./60., max_steps=500, render=False):
        super().__init__()
        self.start = np.array(start, dtype=np.float32)
        self.goal = np.array(goal, dtype=np.float32)
        self.map_size = np.array(map_size, dtype=np.float32)
        self.time_step = time_step
        self.max_steps = max_steps
        self.render_mode = "rgb_array" if render else None

        self.obstacles = obstacles if obstacles is not None else []
        self.action_dim = 2
        self.obs_dim = 4
        # Gym spaces
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.action_dim,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32)

        # PyBullet setup
        self._connect_pybullet(render)
        self._build_scene()
        # --- inside __init__ after building scene ---
        self.path = []                     # list of [x,y] positions for trajectory
        self._start_marker_id = None
        self._goal_marker_id = None
        self._debug_line_ids = []          # store debug line ids so we can remove if needed
        self._create_start_goal_markers()

    # --- new helper to create start and goal markers ---
    def _create_start_goal_markers(self):
        # small spheres for start (green) and goal (red)
        radius = 0.18
        start_vis = p.createVisualShape(p.GEOM_SPHERE, radius=radius, rgbaColor=[0,1,0,1], physicsClientId=self.client)
        goal_vis  = p.createVisualShape(p.GEOM_SPHERE, radius=radius, rgbaColor=[1,0,0,1], physicsClientId=self.client)
        # create static multibodies (mass=0) at z=0.2
        self._start_marker_id = p.createMultiBody(baseMass=0, baseVisualShapeIndex=start_vis,
                                                  basePosition=[self.start[0], self.start[1], 0.2],
                                                  physicsClientId=self.client)
        self._goal_marker_id = p.createMultiBody(baseMass=0, baseVisualShapeIndex=goal_vis,
                                                basePosition=[self.goal[0], self.goal[1], 0.2],
                                                physicsClientId=self.client)

    # --- modify reset to clear path and draw start/goal markers ---
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetBasePositionAndOrientation(self.robot, [self.start[0], self.start[1], 0.2], [0,0,0,1], physicsClientId=self.client)
        self.steps = 0
        self.position = self.start.copy()
        # reset path and debug lines
        self.path = [self.position.copy()]
        for lid in self._debug_line_ids:
            try:
                p.removeUserDebugItem(lid, physicsClientId=self.client)
            except Exception:
                pass
        self._debug_line_ids = []
        # ensure start/goal markers exist and are at correct positions
        if self._start_marker_id is None or self._goal_marker_id is None:
            self._create_start_goal_markers()
        else:
            p.resetBasePositionAndOrientation(self._start_marker_id, [self.start[0], self.start[1], 0.2], [0,0,0,1], physicsClientId=self.client)
            p.resetBasePositionAndOrientation(self._goal_marker_id, [self.goal[0], self.goal[1], 0.2], [0,0,0,1], physicsClientId=self.client)
        return self._get_state(), {}

    # --- optional: function to clear trajectory lines ---
    def clear_trajectory(self):
        for lid in self._debug_line_ids:
            try:
                p.removeUserDebugItem(lid, physicsClientId=self.client)
            except Exception:
                pass
        self._debug_line_ids = []
        self.path = []


    def _connect_pybullet(self, render):
        # Use DIRECT for headless; GUI if render True
        if hasattr(self, "client") and self.client is not None:
            try:
                p.disconnect(self.client)
            except Exception:
                pass
        if render:
            self.client = p.connect(p.GUI)
        else:
            self.client = p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.8)
        p.setTimeStep(self.time_step)

    def _build_scene(self):
        p.resetSimulation(physicsClientId=self.client)
        self.plane = p.loadURDF("plane.urdf", physicsClientId=self.client)
        # robot sphere
        colSphereId = p.createCollisionShape(p.GEOM_SPHERE, radius=0.2, physicsClientId=self.client)
        visSphereId = p.createVisualShape(p.GEOM_SPHERE, radius=0.2, rgbaColor=[0,0,1,1], physicsClientId=self.client)
        self.robot = p.createMultiBody(baseMass=1.0, baseCollisionShapeIndex=colSphereId,
                                       baseVisualShapeIndex=visSphereId,
                                       basePosition=[self.start[0], self.start[1], 0.2],
                                       physicsClientId=self.client)
        # obstacles
        self.obs_ids = []
        for obs in self.obstacles:
            x = (obs["x_min"] + obs["x_max"]) / 2.0
            y = (obs["y_min"] + obs["y_max"]) / 2.0
            sx = (obs["x_max"] - obs["x_min"]) / 2.0
            sy = (obs["y_max"] - obs["y_min"]) / 2.0
            colBoxId = p.createCollisionShape(p.GEOM_BOX, halfExtents=[sx, sy, 0.5], physicsClientId=self.client)
            visBoxId = p.createVisualShape(p.GEOM_BOX, halfExtents=[sx, sy, 0.5], rgbaColor=[0.5,0.5,0.5,1], physicsClientId=self.client)
            obs_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=colBoxId,
                                       baseVisualShapeIndex=visBoxId,
                                       basePosition=[x, y, 0.5], physicsClientId=self.client)
            self.obs_ids.append(obs_id)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        p.resetBasePositionAndOrientation(self.robot, [self.start[0], self.start[1], 0.2], [0,0,0,1], physicsClientId=self.client)
        self.steps = 0
        self.position = self.start.copy()
        return self._get_state(), {}

    def _get_state(self):
        dx, dy = self.goal - self.position
        dist = np.linalg.norm([dx, dy])
        nearest = float('inf')
        for obs in self.obstacles:
            cx = np.clip(self.position[0], obs["x_min"], obs["x_max"])
            cy = np.clip(self.position[1], obs["y_min"], obs["y_max"])
            d = np.linalg.norm(self.position - np.array([cx, cy]))
            if d < nearest:
                nearest = d
        if nearest == float('inf'):
            nearest = max(self.map_size)
        diag = max(1e-6, np.hypot(self.map_size[0], self.map_size[1]))
        dist_norm = dist / diag
        obs = np.array([dx / self.map_size[0], dy / self.map_size[1],dist / np.hypot(*self.map_size), nearest / np.hypot(*self.map_size)], dtype=np.float32)
        #obs = np.array([dx / self.map_size[0], dy / self.map_size[1],dist_norm / np.hypot(*self.map_size), nearest / np.hypot(*self.map_size)], dtype=np.float32)


        return obs

    def step(self, action):
        self.steps += 1
        max_step = 0.5
        action = np.clip(action, -1.0, 1.0)
        delta = action * max_step
        next_pos = self.position + delta
        next_pos[0] = np.clip(next_pos[0], 0.0, self.map_size[0])
        next_pos[1] = np.clip(next_pos[1], 0.0, self.map_size[1])
        collision = self._is_collision(next_pos)
        prev_pos = self.position.copy()

        if not collision:
            p.resetBasePositionAndOrientation(self.robot, [next_pos[0], next_pos[1], 0.2], [0,0,0,1], physicsClientId=self.client)
            self.position = next_pos

        self.path.append(self.position.copy())

        # draw a short debug line segment from prev_pos to current position
        color = [0.0, 0.5, 1.0]  # cyan-ish
        line_id = p.addUserDebugLine([prev_pos[0], prev_pos[1], 0.25],
                                    [self.position[0], self.position[1], 0.25],
                                    lineColorRGB=color,
                                    lineWidth=2.0,
                                    lifeTime=0,  # 0 means persistent until removed
                                    physicsClientId=self.client)
        self._debug_line_ids.append(line_id)

        dist_to_goal = np.linalg.norm(self.goal - self.position)
        done = False
        reward = 0.0
        if dist_to_goal < 2.5:
            reward += 100.0
            done = True
        else:
            prev_pos = self.position - delta if not collision else self.position
            prev_dist = np.linalg.norm(self.goal - prev_pos)
            progress = prev_dist - dist_to_goal
            reward += 100.0 * progress
            reward -= 0.1 * np.linalg.norm(delta)

        nearest = float('inf')
        for obs in self.obstacles:
            cx = np.clip(self.position[0], obs["x_min"], obs["x_max"])
            cy = np.clip(self.position[1], obs["y_min"], obs["y_max"])
            d = np.linalg.norm(self.position - np.array([cx, cy]))
            if d < nearest:
                nearest = d
        if nearest < 1.0:
            reward -= 5.0 * (1.0 - np.clip(nearest, 0.0, 1.0))

        if collision:
            reward -= 50.0
            done = True

        if self.steps >= self.max_steps:
            done = True

        return self._get_state(), float(reward), done, False, {"collision": collision}

    def _is_collision(self, pos):
        for obs in self.obstacles:
            if (obs["x_min"] <= pos[0] <= obs["x_max"] and
                obs["y_min"] <= pos[1] <= obs["y_max"]):
                return True
        return False

    def render(self):
        # return RGB array from a fixed camera
        cam_target = [self.position[0], self.position[1], 0.0]
        cam_pos = [self.position[0] - 3.0, self.position[1] - 3.0, 6.0]
        view_matrix = p.computeViewMatrix(cameraEyePosition=cam_pos, cameraTargetPosition=cam_target,
                                          cameraUpVector=[0,0,1], physicsClientId=self.client)
        proj_matrix = p.computeProjectionMatrixFOV(fov=60, aspect=1.0, nearVal=0.1, farVal=50.0, physicsClientId=self.client)
        w, h, rgb, depth, seg = p.getCameraImage(width=640, height=640, viewMatrix=view_matrix,
                                                 projectionMatrix=proj_matrix, physicsClientId=self.client)
        rgb_arr = np.reshape(rgb, (h, w, 4))[:, :, :3]
        return rgb_arr

    def close(self):
        try:
            p.disconnect(self.client)
        except Exception:
            pass



### PPO helper functions and TRAINING logic

In [106]:
Transition = namedtuple('Transition', ['state', 'action', 'log_prob', 'reward', 'value', 'done'])

def compute_gae(transitions, gamma=0.90, lam=0.95):
    rewards = [t.reward for t in transitions]
    values = [t.value for t in transitions] + [0.0]
    dones = [t.done for t in transitions]
    gae = 0.0
    returns = []
    advantages = []
    for step in reversed(range(len(rewards))):
        mask = 0.0 if dones[step] else 1.0
        delta = rewards[step] + gamma * values[step+1] * mask - values[step]
        gae = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)
        returns.insert(0, gae + values[step])
    adv = torch.tensor(advantages, dtype=torch.float32)
    ret = torch.tensor(returns, dtype=torch.float32)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    return ret, adv

def sanitize_obs(obs):
    obs = np.asarray(obs, dtype=np.float32)
    # replace inf/nan with large finite values or zeros
    obs = np.nan_to_num(obs, nan=0.0, posinf=1e6, neginf=-1e6)
    # clip to reasonable range
    return np.clip(obs, -1e3, 1e3)


# ---------------------------
# Training loop (small by default for demo)
# ---------------------------
# Replace train_ppo with this device-safe version
def train_ppo(env, agent, epochs=40, steps_per_epoch=1024, batch_size=64, clip_eps=0.2, lr=0.05, vf_coef=0.5, ent_coef=0.01):
    optimizer = optim.Adam(agent.parameters(), lr=lr)
    agent.to(device)  # ensure model on device

    for epoch in range(epochs):
        transitions = []
        obs, _ = env.reset()
        ep_rewards = []
        for step in range(steps_per_epoch):
            # prepare obs on device for agent; agent.get_action will ensure device match
            #obs = sanitize_obs(obs)
            action_np, log_prob, value = agent.get_action(obs)
            action_np = np.clip(action_np, -1.0, 1.0)
            next_obs, reward, done, terminated, info = env.step(action_np)

            # store tensors consistently (move state/action to CPU for storage or keep on device)
            # We'll store state/action as CPU tensors and log_prob/value as device tensors
            transitions.append(Transition(
                state=torch.tensor(obs, dtype=torch.float32),  # CPU for stacking later
                action=torch.tensor(action_np, dtype=torch.float32),
                log_prob=log_prob.detach(),  # tensor on device
                reward=reward,
                value=value.detach().cpu().item() if isinstance(value, torch.Tensor) else float(value),
                done=done or terminated
            ))

            obs = next_obs
            ep_rewards.append(reward)
            if done or terminated:
                obs, _ = env.reset()

        # Prepare training tensors and move to device
        states = torch.stack([t.state for t in transitions]).to(device)
        actions = torch.stack([t.action for t in transitions]).to(device)
        # old_log_probs were stored as tensors on device already; stack and ensure device
        old_log_probs = torch.stack([t.log_prob.to(device) for t in transitions]).to(device)
        returns, advantages = compute_gae(transitions)
        returns = returns.to(device)
        advantages = advantages.to(device)

        n = states.size(0)
        idxs = np.arange(n)
        for _ in range(8):
            np.random.shuffle(idxs)
            for start in range(0, n, batch_size):
                batch_idx = idxs[start:start+batch_size]
                b_states = states[batch_idx]
                b_actions = actions[batch_idx]
                b_old_log_probs = old_log_probs[batch_idx]
                b_returns = returns[batch_idx]
                b_advantages = advantages[batch_idx]

                log_probs, entropy, values = agent.evaluate_actions(b_states, b_actions)
                # log_probs, entropy, values are on device because model is on device
                ratios = torch.exp(log_probs - b_old_log_probs)
                surr1 = ratios * b_advantages
                surr2 = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps) * b_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = ((b_returns - values) ** 2).mean()
                entropy_loss = -entropy.mean()
                loss = actor_loss + vf_coef * critic_loss + ent_coef * entropy_loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
                optimizer.step()

        avg_reward = sum(ep_rewards) / max(1, len(ep_rewards))
        print(f"Epoch {epoch+1}/{epochs}  AvgRolloutReward {avg_reward:.2f}")



### Function for Rollout and capture run in the video

In [102]:
def demo_and_record(env, agent, out_path="demo.mp4", max_steps=500): #Change Max_steps to atleast 500 to get near to realistic experiment!!!!
    frames = []
    obs, _ = env.reset()
    for step in range(max_steps):
        action_np, _, _ = agent.get_action(torch.tensor(obs, dtype=torch.float32))
        obs_from = obs
        action=action_np
        action_np = np.clip(action_np, -1.0, 1.0)
        obs, reward, done, terminated, info = env.step(action_np)
        print('Step : '+str(step)+'   obs : '+str(obs_from)+'   action : '+str(action)+'     Clipped ACtion : '+str(action_np)+'     Reward :  '+str(reward)+'    NEXT state : '+str(obs))
        frame = env.render()
        frames.append(frame)
        if done or terminated:
            break
    # write video
    writer = imageio.get_writer(out_path, fps=30)
    for f in frames:
        writer.append_data(f)
    writer.close()
    return out_path



## Main Function to Run

In [138]:
if __name__ == "__main__":
    obstacles = [
        {"x_min": 3.0, "x_max": 5.0, "y_min": 3.0, "y_max": 5.0},
        {"x_min": 6.0, "x_max": 8.0, "y_min": 6.0, "y_max": 8.0}
    ]
    env = PyBulletPointEnv(start=(1.0,1.0), goal=(5.5,5.5), obstacles=obstacles, map_size=(15.0,15.0),
                           time_step=1./240., max_steps=300, render=False)

    obs_dim = env.obs_dim
    action_dim = env.action_dim
    agent = PPOAgent(obs_dim, action_dim, hidden_dim=128)

    # Train (small run for demo). Increase epochs/steps_per_epoch for better policies.
    train_ppo(env, agent, epochs=30, steps_per_epoch=50, batch_size=64)
    #train_ppo(env, agent, epochs=30, steps_per_epoch=512, batch_size=64)

    # Demo and record video
    video_file = "ppo_pybullet_demo.mp4"
    out_path = demo_and_record(env, agent, out_path=video_file, max_steps=100)
    env.close()

    # Display video in Colab
    display(Video(out_path, embed=True, width=640, height=640))

Epoch 1/30  AvgRolloutReward -0.81
Epoch 2/30  AvgRolloutReward 38.58
Epoch 3/30  AvgRolloutReward 41.13
Epoch 4/30  AvgRolloutReward 22.71
Epoch 5/30  AvgRolloutReward 40.97
Epoch 6/30  AvgRolloutReward 40.97
Epoch 7/30  AvgRolloutReward 40.97
Epoch 8/30  AvgRolloutReward 40.97
Epoch 9/30  AvgRolloutReward 40.97
Epoch 10/30  AvgRolloutReward 40.97
Epoch 11/30  AvgRolloutReward 40.97
Epoch 12/30  AvgRolloutReward 40.97
Epoch 13/30  AvgRolloutReward 40.97
Epoch 14/30  AvgRolloutReward 40.97
Epoch 15/30  AvgRolloutReward 40.97
Epoch 16/30  AvgRolloutReward 40.97
Epoch 17/30  AvgRolloutReward 40.97
Epoch 18/30  AvgRolloutReward 40.97
Epoch 19/30  AvgRolloutReward 40.97
Epoch 20/30  AvgRolloutReward 40.97
Epoch 21/30  AvgRolloutReward 40.97
Epoch 22/30  AvgRolloutReward 40.97
Epoch 23/30  AvgRolloutReward 40.97
Epoch 24/30  AvgRolloutReward 40.97
Epoch 25/30  AvgRolloutReward 39.59
Epoch 26/30  AvgRolloutReward 41.41
Epoch 27/30  AvgRolloutReward 41.57
Epoch 28/30  AvgRolloutReward 40.97
E

In [86]:
print('State Dimension: '+str(obs_dim))
print('Action Dimension: '+str(action_dim))

State Dimension: 4
Action Dimension: 2


## Exercises:
1. Rewards are hardcoded in above code. Instead dynamic reward can be designed as a heuristic function considering the dynamic obstacles, Goals et.,

2. Replace the 4‑dim state with a 1D LIDAR array and train the same PPO; Compare sample efficiency and success rate.

3. Train with moving obstacles to learn dynamic avoidance.

4. Use the trained policy as a steering function inside RRT: sample nodes, but use the policy to propose local motions between nodes. Measure planning time and path quality.

5. Compare PPO vs SAC on the same environment and plot learning curves.
